## Converting SMAP L4 GPD to Parquet 

This notebook works to convert L4 SMAP files to parquet files and combine March to July into one df.

In [1]:
import pandas as pd
import numpy as np
import h5py
from pathlib import Path
from tqdm import tqdm

BASE_DIR = Path('/global/scratch/users/rubypercival/SMAP24')
GPD_DIR = BASE_DIR / 'GPD'
OUTPUT_DIR = BASE_DIR / 'parquet'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
gpd_files = sorted(GPD_DIR.rglob('*.h5'))
print(f'Found {len(gpd_files):,} GPD .h5 files')
for f in gpd_files[:5]:
    print(f)


Found 1,225 GPD .h5 files
/global/scratch/users/rubypercival/SMAP24/GPD/SMAP_L4_SM_gph_20240229T223000_Vv8010_001.h5
/global/scratch/users/rubypercival/SMAP24/GPD/SMAP_L4_SM_gph_20240301T013000_Vv8010_001.h5
/global/scratch/users/rubypercival/SMAP24/GPD/SMAP_L4_SM_gph_20240301T043000_Vv8010_001.h5
/global/scratch/users/rubypercival/SMAP24/GPD/SMAP_L4_SM_gph_20240301T073000_Vv8010_001.h5
/global/scratch/users/rubypercival/SMAP24/GPD/SMAP_L4_SM_gph_20240301T103000_Vv8010_001.h5


### Process GPD files

In [3]:
def process_gpd_folder(folder_path, lat_min, lat_max, lon_min, lon_max,
                        time_origin='2000-01-01 11:58:55.816', missing_value=-9999):
    files = sorted(Path(folder_path).rglob('*.h5'))
    df_list = []

    for file in tqdm(files, desc='Processing GPD files'):
        with h5py.File(file, 'r') as f:
            sm_surface = f['Geophysical_Data/sm_surface'][:]
            sm_rootzone = f['Geophysical_Data/sm_rootzone'][:]
            surface_temp = f['Geophysical_Data/surface_temp'][:]
            soil_temp_layer1 = f['Geophysical_Data/soil_temp_layer1'][:]
            LAI = f['Geophysical_Data/leaf_area_index'][:]
            cell_lat = f['cell_lat'][:]
            cell_lon = f['cell_lon'][:]
            time = f['time'][:]

            sm_surface = np.where(sm_surface == missing_value, np.nan, sm_surface)
            sm_rootzone = np.where(sm_rootzone == missing_value, np.nan, sm_rootzone)
            surface_temp = np.where(surface_temp == missing_value, np.nan, surface_temp)
            soil_temp_layer1 = np.where(soil_temp_layer1 == missing_value, np.nan, soil_temp_layer1)
            LAI = np.where(LAI == missing_value, np.nan, LAI)

            df = pd.DataFrame({
                'latitude': cell_lat.ravel(),
                'longitude': cell_lon.ravel(),
                'sm_surface': sm_surface.ravel(),
                'sm_rootzone': sm_rootzone.ravel(),
                'surface_temp': surface_temp.ravel(),
                'soil_temp_layer1': soil_temp_layer1.ravel(),
                'LAI': LAI.ravel(),
                'time': np.full(cell_lat.size, time[0])})
            df['datetime'] = pd.to_datetime(df['time'], unit='s', origin=time_origin)
            df = df.dropna(subset=['sm_surface', 'sm_rootzone', 'surface_temp', 'LAI'])
            df.drop(columns='time', inplace=True)

            region_df = df[
                (df['latitude'] >= lat_min) & (df['latitude'] <= lat_max) &
                (df['longitude'] >= lon_min) & (df['longitude'] <= lon_max)]
            df_list.append(region_df)

In [4]:
lon_min, lon_max = -104, -80
lat_min, lat_max = 36, 49

gpd_df = process_gpd_folder(GPD_DIR, lat_min, lat_max, lon_min, lon_max)
print('GPD shape:', gpd_df.shape)

output_parquet = OUTPUT_DIR / 'SMAP_GPD_mar-jul24.parquet'
gpd_df.to_parquet(output_parquet, engine='pyarrow', index=False)
print('Saved to:', output_parquet)


Processing GPD files: 100%|██████████| 1225/1225 [27:46<00:00,  1.36s/it]


GPD shape: (40217975, 8)
Saved to: /global/scratch/users/rubypercival/SMAP24/parquet/SMAP_GPD_mar-jul24.parquet


In [5]:
df = pd.read_parquet(output_parquet, engine='pyarrow')
print('Shape:', df.shape)
print(df.columns.tolist())
df.head()


Shape: (40217975, 8)
['latitude', 'longitude', 'sm_surface', 'sm_rootzone', 'surface_temp', 'soil_temp_layer1', 'LAI', 'datetime']


,latitude,longitude,sm_surface,sm_rootzone,surface_temp,soil_temp_layer1,LAI,datetime
0,48.951267,-103.957474,0.092959,0.115514,282.461212,273.161682,0.042729,2024-02-29 22:28:55
1,48.951267,-103.864113,0.166994,0.197637,282.129181,273.091370,0.036702,2024-02-29 22:28:55
2,48.951267,-103.770752,0.168032,0.197454,281.304840,272.724182,0.030683,2024-02-29 22:28:55
3,48.951267,-103.677391,0.166872,0.200200,281.550812,272.728088,0.029180,2024-02-29 22:28:55
4,48.951267,-103.584030,0.165163,0.201177,281.975861,272.935120,0.029721,2024-02-29 22:28:55
